In [8]:
import pandas as pd
from transformers import pipeline
import re

In [9]:
data = pd.read_csv("raw_data/dataset.csv")
common_names = pd.read_csv("utils/common_names.csv")
data[["Resume"]].to_csv("raw_data/resumes.csv", index=False)

In [10]:
data["Resume"][0]
pd.DataFrame({"Resume": [data["Resume"][0]]}).to_csv("first_resume.csv", index=False)

In [11]:
gender_words = {
    'female': ['woman', 'girl', 'lady', 'mother', 'wife', 'daughter', 'sister', 'mom'],
    'male': ['man', 'boy', 'boys', 'gentleman', 'father', 'husband', 'son', 'brother', 'dad']
}

In [12]:
ner_pipeline = pipeline("ner",
                        model="ab-ai/pii_model_based_on_distilbert",
                        aggregation_strategy="simple",
                        device=-1)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 558.17it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   


In [ ]:
results = predict_entities_chunked(data["Resume"][0])
for entity in results:
    print(entity)

RuntimeError: The size of tensor a (516) must match the size of tensor b (512) at non-singleton dimension 1

In [ ]:
per_entities = [e for e in sorted(results, key=lambda x: x['start']) 
                if e['entity_group'] == 'PER']  # update 'PER' if label differs

In [ ]:
# Test on one resume to see labels
results = ner_pipeline(data["Resume"][0])
for entity in results:
    print(entity)

In [ ]:
def indentified_name(index, extracted_info_dict):
    if not extracted_info_dict[index].get('name'):
        return False
    return True


----------------------------------------------------
#### Identify all names, anonymize the name in the resume and save it in a dictionary

In [ ]:
def anonymize_names(text, index, extracted_info_dict):
    """ Anonymize all names that is the same as the 
    first one mentioned in the Resume. """
    if not isinstance(text, str):
        return text
    
    results = ner_pipeline(text)
    
    if index not in extracted_info_dict:
        extracted_info_dict[index] = {
            'name': [],
        }
    
    # Sort by start position to find the first name
    per_entities = [e for e in sorted(results, key=lambda x: x['start']) 
                    if e['entity_group'] == 'PER']
    
    if not per_entities:
        return text
    
    # First detected name, add to dictionary
    name = per_entities[0]['word']
    extracted_info_dict[index]['name'].append(name)
    name_parts = set(name.lower().split())  # {'jason', 'jones'}

    for entity in sorted(per_entities, key=lambda x: x['start'], reverse=True):
        if all(part in entity['word'].lower() for part in name_parts):
            text = text[:entity['start']] + '[PERSON]' + text[entity['end']:]
    return text



----------------------------------------------------
#### Identify all emails

In [ ]:
def anonymize_emails(text, index, extracted_info_dict):
    """ Anonymize email addresses in the text. """
    if not isinstance(text, str):
        return text
    
    email_pattern = r'[\w\.-]+@[\w\.-]+'
    front_email_pattern = r'[\w\.-]+@'
    emails = re.findall(email_pattern, text)

    if not indentified_name(index, extracted_info_dict): # no name found
        return text

    if index not in extracted_info_dict:
        extracted_info_dict[index] = {
            'emails': [],
        }
    
    if emails:
        extracted_info_dict[index]['emails'] = []
        for email in emails:
            if email not in extracted_info_dict[index]['emails']:
                extracted_info_dict[index]['emails'] = email # store all unique emails
        text = re.sub(front_email_pattern, '[EMAIL]', text)
    
    return text

In [ ]:
def anonymize_usernames(text, index, extracted_info_dict):
    """ Anonymize usernames in the text, e.g. Bob Smidth -> bobsmidth. """
    if not isinstance(text, str):
        return text
    
    if not indentified_name(index, extracted_info_dict): # no name found
        return text
    
    if index not in extracted_info_dict:
        extracted_info_dict[index] = {
            'username': [],
        }
    
    name = extracted_info_dict[index]['name'][0]
    no_space = re.escape(re.sub(r'[\s-]', '', name))
    dashed = re.escape(name.lower().replace(" ", "-"))
    pattern = re.compile(f"{no_space}|{dashed}", re.IGNORECASE)
    
    if re.search(pattern, text):
        extracted_info_dict[index]['username'] = re.sub(r'[\s-]', '', name).lower()  # store username string
        text = re.sub(pattern, '[USERNAME]', text)

    return text

In [ ]:
def identify_gender(index, extracted_info_dict, common_names):
    """ Identify gender from the first name. """
    
    if not extracted_info_dict[index]['name']:
        extracted_info_dict[index]['gender'] = 'unknown'
        return
    
    if not indentified_name(index, extracted_info_dict): # no name found
        return
    
    full_name = extracted_info_dict[index]['name'][0]
    first_name = full_name.split()[0].lower()
    
    matches = common_names[common_names['Name'].str.lower() == first_name]
    
    if matches.empty:
        extracted_info_dict[index]['gender'] = 'unknown'
        return

    
    male_prob = matches[matches['Gender'] == 'M']['Probability'].sum()
    female_prob = matches[matches['Gender'] == 'F']['Probability'].sum()
    
    if len(matches) == 1:
        extracted_info_dict[index]['gender'] = matches.iloc[0]['Gender']
    else:
        if (female_prob == 0 and male_prob == 0) or len(full_name) <= 2:
            extracted_info_dict[index]['gender'] = 'unknown'
        elif female_prob == 0:
            extracted_info_dict[index]['gender'] = 'M'
        elif male_prob == 0:
            extracted_info_dict[index]['gender'] = 'F'
        elif male_prob / female_prob >= 2:
            extracted_info_dict[index]['gender'] = 'M'
        elif female_prob / male_prob >= 2:
            extracted_info_dict[index]['gender'] = 'F'
        else:
            extracted_info_dict[index]['gender'] = 'unisex'

In [ ]:
def anonymize_gender_words(text, index, extracted_info_dict):
    """ Anonymize gender words that appear next to punctuation or newlines. """
    if not isinstance(text, str):
        return text
    
    extracted_info_dict[index]['gender_words'] = []
    
    for gender, words in gender_words.items():
        for word in words:
            pattern = re.compile(
                r'(?<=[,.\-\n])' + re.escape(word) + r'(?=[,.\-\n]|$)', re.IGNORECASE)
            matches = re.findall(pattern, text)
            if matches:
                extracted_info_dict[index]['gender_words'].extend(matches)
                text = re.sub(pattern, '[GENDER]', text)
    
    return text

### TESTS

In [ ]:
pd.DataFrame({"Resume": data["Resume"][:3]}).to_csv("raw_data/f0_t3_resumes.csv", index=False)
pd.DataFrame({"Resume": data["Resume"][:500]}).to_csv("raw_data/f0_t500_resumes.csv", index=False)
pd.DataFrame({"Resume": data["Resume"][1000:1500]}).to_csv("raw_data/f1000_t1500_resumes.csv", index=False)

In [ ]:
resumes = pd.read_csv("raw_data/resumes.csv")  
extracted_info_dict = {}

# Anonymization
for idx, row in resumes.iterrows():
    text = row['Resume']
    text = anonymize_names(text, idx, extracted_info_dict)
    text = anonymize_emails(text, idx, extracted_info_dict)
    text = anonymize_usernames(text, idx, extracted_info_dict)
    identify_gender(idx, extracted_info_dict, common_names)
    text = anonymize_gender_words(text, idx, extracted_info_dict)
    resumes.at[idx, 'anonymized_text'] = text

In [ ]:

resumes.to_csv("anonymized_files/anonymized_file.csv", index=False)
pd.DataFrame.from_dict(extracted_info_dict, orient='index').to_csv("anonymized_files/dictionary.csv")